# AegisRAG · Stage 04 — DPO Alignment
    6-type Direct Preference Optimisation on top of the SFT adapter from stage 03.
    **Prereq**: upload the stage-03 output as a Kaggle Dataset named `aegisrag-checkpoints` with the layout `aegisrag-checkpoints/generator_sft/<adapter files>` so the bootstrap auto-wires it.
    ~2 h for 1 epoch on 3k preference triplets with batch_size=1, grad_accum=32.


In [ ]:
# ---- Kaggle bootstrap ----------------------------------------------------
# Attach these Kaggle Datasets to the notebook before running:
#   aegisrag-source       (snapshot of the git repo: src/, config/, run.py, requirements.txt)
#   aegisrag-raw-docs     (raw customer-support documents)            [only for data-gen]
#   aegisrag-synthetic    (generated qa/preferences/labels .jsonl)    [all training stages]
#   aegisrag-checkpoints  (prior-stage checkpoints)                   [required for DPO]
import sys, os, importlib.util
spec = importlib.util.spec_from_file_location(
    "aegisrag_bootstrap",
    "/kaggle/input/aegisrag-source/kaggle/helpers/bootstrap.py",
)
bootstrap = importlib.util.module_from_spec(spec); spec.loader.exec_module(bootstrap)
bootstrap.setup_kaggle(
    component='dpo',
    install_train_deps=True,
    install_parse_deps=False,
)


In [ ]:
# DPO memory is higher than SFT (needs reference model). Shrink further.
import os
os.environ["AEGIS_TRAINING__DPO__MAX_SEQ_LENGTH"] = "1024"
os.environ["AEGIS_TRAINING__DPO__MAX_PROMPT_LENGTH"] = "512"
os.environ["AEGIS_TRAINING__DPO__BATCH_SIZE"] = "1"
os.environ["AEGIS_TRAINING__DPO__GRADIENT_ACCUMULATION_STEPS"] = "32"

from src.utils.config import reset_config
reset_config()


In [ ]:
from src.training.train_dpo import train
result = train()
result


In [ ]:
# ---- Persist outputs -----------------------------------------------------
# Everything under /kaggle/working is captured as the notebook's output and
# can be saved as a new Kaggle Dataset via File > "Save Version"
# with output = "Save & Run All (Commit)".
import shutil, os
from pathlib import Path
src_dir = Path("/kaggle/working/checkpoints/dpo")
print("DPO adapter:", src_dir, "->", os.listdir(src_dir) if src_dir.exists() else "(missing)")
